**Portfolio generator**


In [1]:
# imports

import os
from dotenv import load_dotenv
from IPython.display import display
from openai import OpenAI
import ipywidgets as widgets
from IPython.display import display

# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

openai = OpenAI()


API key found and looks good so far!


In [3]:
upload = widgets.FileUpload(
	accept=".pdf",
	multiple=False,
	description="Upload Resume (PDF)"
)

status = widgets.Output()
display(widgets.VBox([upload, status]))

def on_upload_change(change):
	with status:
		status.clear_output()
		if not upload.value:
			print("No file uploaded yet.")
			return

		# ipywidgets v8+: upload.value is a tuple of file info dicts
		# ipywidgets v7:  upload.value is a dict keyed by filename
		if isinstance(upload.value, dict):
			file_info = next(iter(upload.value.values()))
		else:
			file_info = upload.value[0]

		filename = file_info["name"]

		if not filename.lower().endswith(".pdf"):
			print("Please upload a PDF file.")
			return

		os.makedirs("uploads", exist_ok=True)
		resume_path = os.path.join("uploads", filename)

		with open(resume_path, "wb") as f:
			f.write(file_info["content"])

		print(f"Resume uploaded successfully: {resume_path}")

upload.observe(on_upload_change, names="value")

In [5]:
import io
import json
from pypdf import PdfReader

# 1) Read uploaded PDF bytes from the existing upload widget
if not upload.value:
	raise ValueError("No file uploaded. Please upload a resume PDF first.")

file_info = next(iter(upload.value.values())) if isinstance(upload.value, dict) else upload.value[0]
pdf_bytes = bytes(file_info["content"])

# 2) Extract text from PDF
reader = PdfReader(io.BytesIO(pdf_bytes))
resume_text = "\n".join((page.extract_text() or "") for page in reader.pages).strip()

if not resume_text:
	raise ValueError("Could not extract text from this PDF. It may be scanned/image-only.")

# 3) Ask the model to convert resume text into structured JSON
resume_messages = [
	{
		"role": "system",
		"content": (
			"You extract resume data and return only valid JSON. "
			"No markdown, no explanation."
		),
	},
	{
		"role": "user",
		"content": (
			"Convert this resume into JSON with keys: "
			"name, email, phone, location, summary, skills (list), "
			"experience (list of {company, title, start_date, end_date, responsibilities}), "
			"education (list of {institution, degree, field, start_date, end_date}), "
			"certifications (list), projects (list), links (list).\n\n"
			f"Resume text:\n{resume_text}"
		),
	},
]

resume_response = openai.chat.completions.create(
	model="gpt-5-nano",
	messages=resume_messages
)

raw = resume_response.choices[0].message.content.strip()

# Handle accidental code fences if present
if raw.startswith("```"):
	raw = raw.strip("`")
	raw = raw.replace("json", "", 1).strip()

resume_json = json.loads(raw)

print(json.dumps(resume_json, indent=2))

{
  "name": "Talha Abbasi",
  "email": "talhaabbasi1997@gmail.com",
  "phone": "+92 341-3542008",
  "location": "Karachi, Pakistan",
  "summary": "Software engineer with 6+ years of backend, cloud, and frontend development across AWS, IaC, and modern JS frameworks. Proficient in Node.js, NestJS, Python, Go, React ecosystems, and database design with SQL/NoSQL. Experienced in SOC2 readiness, mentoring, and modernizing legacy codebases.",
  "skills": [
    "JavaScript",
    "TypeScript",
    "Go",
    "Python",
    "C#",
    "Node.js",
    "NestJS",
    "Express",
    "Flask",
    "ASP.NET",
    "ReactJS",
    "Next.js",
    "GatsbyJS",
    "Docker",
    "GitHub Actions",
    "AWS CDK",
    "Pulumi",
    "Redis",
    "MySQL",
    "DynamoDB",
    "MongoDB",
    "MS SQL Server",
    "GraphQL",
    "Jest",
    "Cypress",
    "AWS",
    "Lambda",
    "ECS",
    "RDS",
    "API Gateway",
    "React Native"
  ],
  "experience": [
    {
      "company": "metal.so",
      "title": "Senior Softwa

In [6]:
system_prompt = """
You are a professional portfolio strategist, UX-minded content writer, and web copy assistant.

Your task is to transform structured resume data provided in JSON into clear, modern, and visually appealing portfolio website content.

## Goals

- Generate a complete portfolio content structure
- Prioritize clarity, hierarchy, and readability
- Present the candidate as impressive, professional, and truthful
- Optimize for recruiters and hiring managers who scan quickly

## Output Format

Return only valid JSON with the following fields:

{
"hero": { "name": "", "title": "", "tagline": "", "call_to_action": "" },
"about": { "summary": "", "highlights": [] },
"experience": [ { "role": "", "company": "", "duration": "", "achievements": [] } ],
"projects": [ { "name": "", "description": "", "impact": "", "technologies": [], "link": "" } ],
"skills": { "categories": [ { "name": "", "skills": [] } ] },
"education": [ { "institution": "", "degree": "", "year": "" } ],
"extras": [ { "title": "", "description": "" } ],
"theme": { "style": "", "color_palette": [], "tone": "" }
}

## Instructions

- Rewrite content to be concise, impactful, and professional
- Use strong action verbs
- Quantify impact only when it is clearly supported by the input
- Highlight relevant technical strengths and real-world outcomes
- Remove redundant or low-value information
- Prioritize recent and relevant experience
- Keep descriptions short and scannable

## Design Guidance

- Structure content for a modern portfolio website
- Favor a clean, minimal, readable presentation
- Use clear hierarchy: hero, about, projects, experience, skills, education
- Suggest a cohesive theme that matches the candidate profile

## Constraints

- Do not invent fake experience, projects, links, or metrics
- If data is missing, keep fields minimal or empty
- Keep the output concise and ready for rendering
- Return JSON only, with no markdown or explanation

You are crafting a strong professional narrative from the provided resume data.
"""

portfolio_messages = [
{"role": "system", "content": system_prompt},
{"role": "user", "content": json.dumps(resume_json, indent=2)}
]

portfolio_response = openai.chat.completions.create(
model="gpt-5-nano",
messages=portfolio_messages
)

portfolio_response_json = portfolio_response.choices[0].message.content


In [ ]:
raw_portfolio = portfolio_response_json.strip()

# Remove accidental markdown code fences
if raw_portfolio.startswith("```"):
	raw_portfolio = raw_portfolio.strip("`")
	if raw_portfolio.lower().startswith("json"):
		raw_portfolio = raw_portfolio[4:].strip()

# Parse JSON safely
try:
	portfolio_json = json.loads(raw_portfolio)
except json.JSONDecodeError as e:
	raise ValueError(f"Invalid JSON returned by model: {e}")

# Validate top-level keys
required_keys = {
	"hero", "about", "experience", "projects",
	"skills", "education", "extras", "theme"
}
missing = required_keys - set(portfolio_json.keys())
if missing:
	raise ValueError(f"Missing required keys: {sorted(missing)}")

# Pretty-print for inspection
formatted_portfolio_json = json.dumps(portfolio_json, indent=2, ensure_ascii=False)
print(formatted_portfolio_json)


AttributeError: 'str' object has no attribute 'choices'